In [7]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib

In [8]:
filepath = "../data/interim/customer_churn_clean.csv"  
df = pd.read_csv(filepath)

In [ ]:
# Yahan parameter ka naam 'df' hi rakh diya hai
def build_domain_features(df):
    df = df.copy()
    print("Engineering domain features...")
    
    # 1. Tenure Cohorts / Bins
    bins = [-1, 12, 24, 48, 72]
    labels = ['0-12_months', '13-24_months', '25-48_months', '49-72_months']
    df['Tenure_Cohort'] = pd.cut(df['tenure'], bins=bins, labels=labels)
    
    # 2. Financial Engineering Ratios (+1 for safe division)
    df['Monthly_to_Total_Ratio'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1)
    df['Charges_per_Tenure'] = df['TotalCharges'] / (df['tenure'] + 1)
    
    # 3. Add-on Service Counter
    service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
    existing_services = [col for col in service_cols if col in df.columns]
    
    if existing_services:
        df['Service_Count'] = df[existing_services].isin(['Yes']).sum(axis=1)
    else:
        df['Service_Count'] = 0
        
    print(f"Features created! New Shape: {df.shape}")
    return df

# Ab jab aapke paas 'df' pehle se maujud hai, toh yeh seedha run ho jayega:
df_featured = build_domain_features(df)

Engineering domain features...
Features created! New Shape: (7043, 24)


In [11]:
from sklearn.model_selection import train_test_split

def perform_stratified_split(df, target_column='Churn'):
    X = df.drop(columns=[target_column])
    y = df[target_column]
    
    # Stratify ensure karega ke Train aur Test dono me Churn ratio balance rahe
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    
    print(f"X_train Shape: {X_train.shape} | X_test Shape: {X_test.shape}")
    return X_train, X_test, y_train, y_test

# Run karne ke liye:
X_train, X_test, y_train, y_test = perform_stratified_split(df_featured)


X_train Shape: (5634, 23) | X_test Shape: (1409, 23)


In [12]:
def execute_preprocessing_pipeline(X_tr, X_te):
    # Dynamic types categorization
    num_cols = X_tr.select_dtypes(include=['int64', 'float64']).columns.tolist()
    cat_cols = X_tr.select_dtypes(include=['object', 'category']).columns.tolist()
    
    # Drop unneeded ID fields if present
    for identifier in ['customerID', 'id']:
        if identifier in num_cols: num_cols.remove(identifier)
        if identifier in cat_cols: cat_cols.remove(identifier)
            
    # Sub-Pipelines Definition
    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    cat_pipeline = Pipeline([
        ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
    ])
    
    # ColumnTransformer Setup
    preprocessor = ColumnTransformer(transformers=[
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols)
    ], remainder='drop')
    
    print("Fitting preprocessor exclusively on X_train...")
    X_train_proc = preprocessor.fit_transform(X_tr)
    
    print("Transforming X_test (No Fitting!)...")
    X_test_proc = preprocessor.transform(X_te)
    
    # Extract exact features column names mapping
    encoded_cats = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols)
    all_features = num_cols + list(encoded_cats)
    
    X_train_final = pd.DataFrame(X_train_proc, columns=all_features)
    X_test_final = pd.DataFrame(X_test_proc, columns=all_features)
    
    return X_train_final, X_test_final, preprocessor

# Run karne ke liye:
X_train_clean, X_test_clean, preprocessor_obj = execute_preprocessing_pipeline(X_train, X_test)


C:\Users\Administrator\AppData\Local\Temp\ipykernel_13224\3862969687.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_tr.select_dtypes(include=['object', 'category']).columns.tolist()


Fitting preprocessor exclusively on X_train...
Transforming X_test (No Fitting!)...


In [24]:

from pathlib import Path

# Project root: notebooks folder ka parent
PROJECT_ROOT = Path.cwd().parent

def save_pipeline_outputs(X_tr_f, X_te_f, y_tr, y_te, preprocessor):
    # Correct project-level directories
    processed_dir = PROJECT_ROOT / "data" / "processed"
    preprocessor_dir = PROJECT_ROOT / "artifacts" / "preprocessors"

    processed_dir.mkdir(parents=True, exist_ok=True)
    preprocessor_dir.mkdir(parents=True, exist_ok=True)

    # Save datasets
    X_tr_f.to_csv(processed_dir / "X_train.csv", index=False)
    X_te_f.to_csv(processed_dir / "X_test.csv", index=False)
    y_tr.to_csv(processed_dir / "y_train.csv", index=False)
    y_te.to_csv(processed_dir / "y_test.csv", index=False)

    # Save preprocessor
    joblib.dump(
        preprocessor,
        preprocessor_dir / "preprocessor.joblib"
    )

    print("All outputs saved successfully!")
    print("Processed data:", processed_dir)
    print("Preprocessor:", preprocessor_dir)

# Run
save_pipeline_outputs(
    X_train_clean, X_test_clean,
    y_train, y_test, preprocessor_obj
)


All outputs saved successfully!
Processed data: c:\Users\Administrator\customer-churn-intelligence-platform\data\processed
Preprocessor: c:\Users\Administrator\customer-churn-intelligence-platform\artifacts\preprocessors
